In [ ]:
import os, time
from pathlib import Path
import torch
import psutil
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from IPython.display import display, Markdown

%matplotlib inline

display(Markdown("## Part 0 — Setup"))
display(Markdown("_Imports, helpers, RAM tracker, and the model ID. Nothing is downloaded or loaded yet._"))

plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["font.size"] = 10

CATEGORY_COLORS = {
    "weights":          "#d62728",
    "tokenizer":        "#2ca02c",
    "model-config":     "#1f77b4",
    "metadata/license": "#9467bd",
    "other":            "#7f7f7f",
}

PROC = psutil.Process(os.getpid())
def ram_mb(): return PROC.memory_info().rss / (1024 ** 2)

def fmt_size(b):
    if b < 1024:        return f"{b} B"
    if b < 1024**2:     return f"{b/1024:.2f} KB"
    if b < 1024**3:     return f"{b/1024**2:.2f} MB"
    return f"{b/1024**3:.2f} GB"

STAGE_LOG = []

class Stage:
    """Reusable timing + RAM tracker. Use in every level."""
    def __init__(self, name, reads=""):
        self.name, self.reads = name, reads
    def __enter__(self):
        print(f"▶ {self.name}")
        if self.reads: print(f"  reads: {self.reads}")
        self.ram_start = ram_mb()
        self.t_start = time.time()
        return self
    def __exit__(self, *a):
        dt = time.time() - self.t_start
        ram_end = ram_mb()
        delta = ram_end - self.ram_start
        STAGE_LOG.append({
            "stage":         self.name,
            "reads":         self.reads,
            "ram_before_MB": round(self.ram_start, 1),
            "ram_after_MB":  round(ram_end, 1),
            "ram_delta_MB":  round(delta, 1),
            "time_s":        round(dt, 2),
        })
        print(f"  ✓ {dt:.2f}s   ΔRAM +{delta:.1f} MB   total {ram_end:.1f} MB\n")

MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"

print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")
print("@                PART 00: SETUP COMPLETE                   @")
print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")
print("========== ENVIRONMENT ==========")
print("model id   :", MODEL_ID)
print("torch ver  :", torch.__version__)
print("cuda avail :", torch.cuda.is_available())
print("baseline RAM (notebook only) :", f"{ram_mb():.1f} MB")
print("=================================")

In [ ]:
display(Markdown("## Part 1 — Download stage"))
display(Markdown("_Where do files land on disk? What is fetched and in what order?_"))

from huggingface_hub import snapshot_download, scan_cache_dir

print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")
print("@               PART 01: DOWNLOAD STAGE                    @")
print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")

def cache_info(model_id):
    try:
        for r in scan_cache_dir().repos:
            if r.repo_id == model_id:
                return True, r.size_on_disk
    except Exception: pass
    return False, 0

cached_before, size_before = cache_info(MODEL_ID)
display(Markdown(f"**Cache before:** hit=`{cached_before}`" +
                 (f", size=`{fmt_size(size_before)}`" if cached_before else "")))

print("========== STEP 1: CACHE CHECK ==========")
print("cache hit :", cached_before)
if cached_before:
    print("cached size :", fmt_size(size_before))
print("=========================================")

print()
print("========== STEP 2: SNAPSHOT DOWNLOAD ==========")
with Stage("snapshot_download", reads="huggingface hub"):
    local_path = snapshot_download(repo_id=MODEL_ID)
print("local path :", local_path)
print("===============================================")

hf_root = Path(os.path.expanduser("~/.cache/huggingface/hub"))
display(Markdown(f"""
**Cache layout**
- root: `{hf_root}`
- snapshot: `{local_path}`
- relative: `{Path(local_path).relative_to(hf_root)}`
"""))

def categorize(name):
    n = name.lower()
    if n.endswith((".safetensors", ".bin")):                return "weights"
    if n in ("config.json", "generation_config.json"):      return "model-config"
    if "tokenizer" in n or n == "special_tokens_map.json":  return "tokenizer"
    if n.endswith((".md", ".txt")):                         return "metadata/license"
    return "other"

rows = []
for f in os.listdir(local_path):
    full = os.path.join(local_path, f)
    if os.path.isfile(full) or os.path.islink(full):
        s = os.path.getsize(full)
        rows.append({"filename": f, "category": categorize(f),
                     "size_bytes": s, "size": fmt_size(s)})

df_files = pd.DataFrame(rows).sort_values("size_bytes").reset_index(drop=True)
df_files.index = range(1, len(df_files) + 1)
df_files.index.name = "#"

display(Markdown("**Files on disk (small → large = fetch order)**"))
display(df_files[["filename", "category", "size"]])

print()
print("========== STEP 3: FILES ON DISK ==========")
print(f"file count : {len(df_files)}")
print(f"total size : {fmt_size(df_files['size_bytes'].sum())}")
for _, row in df_files.iterrows():
    print(f"  {row['category']:18s} {row['size']:>12s}  {row['filename']}")
print("===========================================")

# horizontal bar — file sizes (log scale)
fig, ax = plt.subplots(figsize=(10, max(3, len(df_files) * 0.4)))
ax.barh(df_files["filename"], df_files["size_bytes"],
        color=[CATEGORY_COLORS[c] for c in df_files["category"]])
ax.set_xscale("log")
ax.set_xlabel("size (bytes, log scale)")
ax.set_title("File sizes on disk")
ax.invert_yaxis()
cats_present = df_files["category"].unique()
ax.legend(handles=[Patch(facecolor=CATEGORY_COLORS[c], label=c) for c in cats_present],
          loc="lower right")
plt.tight_layout(); plt.show()

# # pie — disk usage by category
# cat_sizes = df_files.groupby("category")["size_bytes"].sum().sort_values(ascending=False)
# fig, ax = plt.subplots(figsize=(6, 6))
# ax.pie(cat_sizes.values, labels=cat_sizes.index,
#        colors=[CATEGORY_COLORS[c] for c in cat_sizes.index],
#        autopct=lambda p: f"{p:.1f}%\n({fmt_size(p * cat_sizes.sum() / 100)})",
#        startangle=90)
# ax.set_title("Disk usage by category"); plt.tight_layout(); plt.show()

total = df_files["size_bytes"].sum()
display(Markdown(f"""
**Summary** — files: `{len(df_files)}`, total: `{fmt_size(total)}`, fresh: `{not cached_before}`
"""))

In [ ]:
display(Markdown("## Part 2 — Load into RAM stage"))
display(Markdown("_Disk → RAM. What is read, in what order, how much memory is consumed?_"))

from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM

print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")
print("@              PART 02: LOAD INTO RAM STAGE                @")
print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")

ram_baseline = ram_mb()
display(Markdown(f"**Baseline RAM (before model load):** `{ram_baseline:.1f} MB`"))

print("========== STEP 1: BASELINE RAM ==========")
print(f"RAM before any load : {ram_baseline:.1f} MB")
print("==========================================")

log_start = len(STAGE_LOG)

print()
print("========== STEP 2: LOAD TOKENIZER ==========")
print("reads : tokenizer.json, tokenizer_config.json, special_tokens_map.json")
with Stage("load tokenizer", reads="tokenizer.json, tokenizer_config.json, special_tokens_map.json"):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
print("tokenizer class :", type(tokenizer).__name__)
print("vocab size      :", tokenizer.vocab_size)
print("============================================")

print()
print("========== STEP 3: LOAD MODEL CONFIG ==========")
print("reads : config.json")
with Stage("load model config", reads="config.json"):
    config = AutoConfig.from_pretrained(MODEL_ID)
print("config class :", type(config).__name__)
print("architecture :", config.architectures[0] if config.architectures else "n/a")
print("===============================================")

print()
print("========== STEP 4: LOAD MODEL WEIGHTS ==========")
print("reads : *.safetensors")
with Stage("load model weights", reads="*.safetensors"):
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
    model.eval()
print("model class : ", type(model).__name__)
print("model dtype : ", next(model.parameters()).dtype)
print("model device: ", next(model.parameters()).device)
print("================================================")

df_load = pd.DataFrame(STAGE_LOG[log_start:])
display(Markdown("**Step-by-step load log**"))
display(df_load)

# RAM growth bar
labels  = ["baseline"] + df_load["stage"].tolist()
values  = [ram_baseline] + df_load["ram_after_MB"].tolist()
deltas  = [0] + df_load["ram_delta_MB"].tolist()
colors  = ["#7f7f7f", "#2ca02c", "#1f77b4", "#d62728"]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(labels, values, color=colors)
for bar, d in zip(bars, deltas):
    h = bar.get_height()
    txt = f"{h:.0f} MB" + (f"\n(+{d:.0f})" if d > 0 else "")
    ax.text(bar.get_x() + bar.get_width()/2, h, txt, ha="center", va="bottom", fontsize=9)
ax.set_ylabel("RAM (MB)"); ax.set_title("RAM growth across load steps")
ax.set_ylim(0, max(values) * 1.15)
plt.tight_layout(); plt.show()

# time per step
fig, ax = plt.subplots(figsize=(10, 3))
ax.barh(df_load["stage"], df_load["time_s"], color=["#2ca02c", "#1f77b4", "#d62728"])
for i, v in enumerate(df_load["time_s"]):
    ax.text(v, i, f"  {v:.2f} s", va="center")
ax.set_xlabel("seconds"); ax.set_title("Time per load step")
plt.tight_layout(); plt.show()

total_params = sum(p.numel() for p in model.parameters())
param_bytes  = sum(p.numel() * p.element_size() for p in model.parameters())

print()
print("========== STEP 5: MODEL SUMMARY ==========")
print(f"architecture     : {config.architectures[0] if config.architectures else 'n/a'}")
print(f"hidden size      : {config.hidden_size}")
print(f"num layers       : {config.num_hidden_layers}")
print(f"vocab size       : {tokenizer.vocab_size}")
print(f"dtype            : {next(model.parameters()).dtype}")
print(f"device           : {next(model.parameters()).device}")
print(f"total parameters : {total_params:,}  ({total_params / 1e9:.3f} B)")
print(f"theoretical size : {fmt_size(param_bytes)}")
print(f"RAM by model     : {(ram_mb() - ram_baseline):.1f} MB")
print("===========================================")

display(Markdown(f"""
**Model summary**

| property | value |
|---|---|
| architecture     | `{config.architectures[0] if config.architectures else 'n/a'}` |
| hidden size      | `{config.hidden_size}` |
| num layers       | `{config.num_hidden_layers}` |
| vocab size       | `{tokenizer.vocab_size}` |
| dtype            | `{next(model.parameters()).dtype}` |
| device           | `{next(model.parameters()).device}` |
| total parameters | `{total_params:,}` ({total_params / 1e9:.3f} B) |
| theoretical size | `{fmt_size(param_bytes)}` |
| RAM by model     | `{(ram_mb() - ram_baseline):.1f} MB` |
"""))

In [ ]:
enc = tokenizer.apply_chat_template([{"role":"user","content":"hi"}], add_generation_prompt=True, return_tensors="pt")
type(enc), (enc if torch.is_tensor(enc) else enc["input_ids"]).shape

In [ ]:
# ---------------------------------------------------------------------------
# Level 10 — one inference, one token. 8 steps exposed; LLM core opaque.
# Assumes the following already exist from your setup:
#   tokenizer, model (fp16, .eval()), config, Stage, STAGE_LOG, fmt_size, ram_mb
# ---------------------------------------------------------------------------
import torch
import pandas as pd

# --- shape constants pulled from the live config (1B values, not the 8B doc) ---
d          = config.hidden_size          # 2048
d_ff       = config.intermediate_size    # 8192
n_h        = config.num_attention_heads  # 32
n_kv       = config.num_key_value_heads  # 8
d_h        = d // n_h                     # 64
N          = config.num_hidden_layers    # 16
vocab_size = config.vocab_size           # 128256

DEVICE = next(model.parameters()).device
WEIGHTS_FILE = "model.safetensors (resident in RAM/VRAM after load; not re-read per step)"

# tied-weight fact: lm_head and embed_tokens share one tensor in 1B
TIED = model.lm_head.weight.data_ptr() == model.model.embed_tokens.weight.data_ptr()

# ---------------------------------------------------------------------------
# Per-step recorder. Captures the 7 fields you asked for, plus time/ram via Stage.
# ---------------------------------------------------------------------------
STEP_LOG = []

def shp(x):
    if isinstance(x, torch.Tensor):
        return "scalar" if x.dim() == 0 else "[" + " x ".join(map(str, x.shape)) + "]"
    if isinstance(x, str):
        return "string"
    return str(type(x).__name__)

def record(num, name, in_name, in_obj, out_name, out_obj, process, files, calc):
    STEP_LOG.append({
        "#":          num,
        "step":       name,
        "input":      in_name,
        "in_shape":   shp(in_obj),
        "output":     out_name,
        "out_shape":  shp(out_obj),
        "process":    process,
        "files":      files,
        "calculation": calc,
    })

# ---------------------------------------------------------------------------
@torch.no_grad()
def run_level10(prompt: str):
    STEP_LOG.clear()

    # --- 1. tokenize -------------------------------------------------------
    with Stage("1 tokenize", reads="tokenizer.json, tokenizer_config.json, special_tokens_map.json"):
        messages = [{"role": "user", "content": prompt}]
        enc = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt"
        )
        # version-robust: some transformers versions return a BatchEncoding dict,
        # others return a bare tensor.
        input_ids = (enc if torch.is_tensor(enc) else enc["input_ids"]).to(DEVICE)  # [1, N_tok]
    record(1, "tokenize", "text", prompt, "token IDs", input_ids,
           "BPE byte-pair merge, map pieces to integer IDs, wrap with chat/special tokens",
           "tokenizer.json, tokenizer_config.json, special_tokens_map.json",
           "deterministic table merges + ID lookup; no floating-point math")
    N_tok = input_ids.shape[1]

    # --- 2. embed (row lookup in {E}) -------------------------------------
    with Stage("2 embed", reads="model.embed_tokens.weight"):
        embeds = model.model.embed_tokens(input_ids)  # [1, N_tok, d]
    record(2, "embed", "token IDs", input_ids, "vectors", embeds,
           "row lookup in {E}: row i is the learned vector for token id i",
           f"{WEIGHTS_FILE}  ->  embed_tokens.weight [{vocab_size} x {d}]",
           "out[b,t] = E[ ids[b,t] ]  (indexing, NOT a matmul)")

    # --- 3. LLM core (OPAQUE: rotary + 16 layers + final RMSNorm) ---------
    with Stage("3 LLM core (opaque)", reads="all layer weights + model.norm.weight"):
        core = model.model(inputs_embeds=embeds, use_cache=False)
        hidden = core.last_hidden_state                # [1, N_tok, d], final-normed
    record(3, "LLM core (opaque)", "vectors", embeds, "vectors", hidden,
           "BLACK BOX: rotary pos emb -> 16 decoder layers (attn+FFN, residual, pre-norm) -> final RMSNorm",
           f"config.json (shapes) + {WEIGHTS_FILE} (16 layers + model.norm.weight)",
           "opaque at L10; only invariant asserted: shape in == shape out, now contextualized")

    # --- 4. LM head (matmul with {H}, tied to {E}) ------------------------
    with Stage("4 LM head", reads="lm_head.weight (tied to embed_tokens.weight)"):
        logits_all = model.lm_head(hidden)             # [1, N_tok, vocab_size]
    record(4, "LM head", "vectors", hidden, "logits (all pos)", logits_all,
           f"linear projection onto vocab with {{H}} (TIED to {{E}}: {TIED})",
           f"{WEIGHTS_FILE}  ->  lm_head.weight [{d} x {vocab_size}] (same tensor as embed)",
           "logits = hidden @ H^T ; each elem = dot(d-vec hidden, d-vec vocab row)")

    # --- 5. take last row -------------------------------------------------
    with Stage("5 take last row", reads="(none)"):
        last_logits = logits_all[:, -1, :]             # [1, vocab_size]
    record(5, "take last row", "logits (all pos)", logits_all, "logits (last pos)", last_logits,
           "slice sequence axis at index -1; only last position predicts the next token",
           "(none) - pure tensor op",
           "out = logits[:, -1, :] ; index selection, no arithmetic")

    # --- 6. softmax (done in fp32 for numerical stability) ----------------
    with Stage("6 softmax", reads="(none)"):
        probs = torch.softmax(last_logits.float(), dim=-1)  # [1, vocab_size]
    record(6, "softmax", "logits (last pos)", last_logits, "probabilities", probs,
           "softmax over vocab axis (cast to fp32 first for stability)",
           "(none)",
           "p_i = exp(z_i - max z) / sum_j exp(z_j - max z) ; sums to 1")

    # --- 7. pick = argmax -------------------------------------------------
    with Stage("7 pick (argmax)", reads="(none)"):
        next_id = torch.argmax(probs, dim=-1)          # [1]
    record(7, "pick (argmax)", "probabilities", probs, "token ID", next_id[0],
           "greedy: index of the maximum probability",
           "(none)",
           "id = argmax_i p_i ; deterministic")

    # --- 8. decode --------------------------------------------------------
    with Stage("8 decode", reads="tokenizer.json"):
        text_piece = tokenizer.decode(next_id)
    record(8, "decode", "token ID", next_id[0], "text piece", text_piece,
           "inverse vocab lookup id -> byte fragment, byte-level BPE reassembly",
           "tokenizer.json (vocab + merges)",
           "table lookup + byte decode; no floating-point math")

    return next_id.item(), text_piece

# ---------------------------------------------------------------------------
if __name__ == "__main__":
    tok_id, piece = run_level10("The capital of France is")

    df_steps = pd.DataFrame(STEP_LOG).set_index("#")
    pd.set_option("display.max_colwidth", None)
    pd.set_option("display.width", None)

    print(f"\nsizes: d={d} d_ff={d_ff} n_h={n_h} n_kv={n_kv} d_h={d_h} "
          f"N={N} vocab={vocab_size}  tied_embeddings={TIED}\n")
    print(df_steps[["step", "input", "in_shape", "output", "out_shape"]], "\n")
    print(df_steps[["step", "process"]], "\n")
    print(df_steps[["step", "files"]], "\n")
    print(df_steps[["step", "calculation"]], "\n")
    print(f"predicted token id = {tok_id!r}   text = {piece!r}")

In [ ]:
import pandas as pd

steps = pd.DataFrame(STEP_LOG)

stage = pd.DataFrame(STAGE_LOG)
stage["#"] = stage["stage"].str.extract(r"^(\d+)\s")
stage = stage.dropna(subset=["#"]).astype({"#": int})
timing = (stage[["#", "time_s", "ram_delta_MB"]]
          .drop_duplicates(subset="#", keep="last"))

df = (steps.merge(timing, on="#", how="left").set_index("#")
      [["step", "input", "in_shape", "output", "out_shape",
        "process", "files", "calculation", "time_s", "ram_delta_MB"]])
assert df.index.is_unique, "index still non-unique -> check STEP_LOG/STAGE_LOG"

display(
    df.style
      .set_properties(**{"text-align": "left", "vertical-align": "top"})
      .set_properties(subset=["process", "files", "calculation"],
                      **{"white-space": "pre-wrap", "max-width": "420px"})
      .set_table_styles([
          {"selector": "th", "props": [("text-align", "left"),
                                       ("vertical-align", "top"), ("padding", "6px")]},
          {"selector": "td", "props": [("padding", "6px"),
                                       ("border", "1px solid #ddd")]},
      ])
      .apply(lambda r: ["background-color:#3a3a3a;color:#f0f0f0"
                        if r.name == 3 else "" for _ in r], axis=1)
)

In [ ]:
# ---------------------------------------------------------------------------
# Level 10 — one inference, one token. 8 steps exposed; LLM core opaque.
# Assumes from setup: tokenizer, model (fp16, .eval()), config,
#                     Stage, STAGE_LOG, fmt_size, ram_mb
# ---------------------------------------------------------------------------
import torch
import pandas as pd

# --- shape constants from live config (1B values) ---
d          = config.hidden_size          # 2048
d_ff       = config.intermediate_size    # 8192
n_h        = config.num_attention_heads  # 32
n_kv       = config.num_key_value_heads  # 8
d_h        = d // n_h                     # 64
N          = config.num_hidden_layers    # 16
vocab_size = config.vocab_size           # 128256

DEVICE = next(model.parameters()).device
WEIGHTS_FILE = "model.safetensors (resident in RAM/VRAM after load; not re-read per step)"

# tied-weight fact: lm_head and embed_tokens share one tensor in 1B
TIED = model.lm_head.weight.data_ptr() == model.model.embed_tokens.weight.data_ptr()

# CHOOSE TRACE MODE:
#   "completion" -> raw next-token continuation (matches "The capital of France is" -> " Paris")
#   "chat"       -> instruct response (predicts first token of assistant reply)
TRACE_MODE = "completion"

STEP_LOG = []

def shp(x):
    if isinstance(x, torch.Tensor):
        return "scalar" if x.dim() == 0 else "[" + " x ".join(map(str, x.shape)) + "]"
    if isinstance(x, str):
        return "string"
    return str(type(x).__name__)

def record(num, name, in_name, in_obj, out_name, out_obj, process, files, calc):
    STEP_LOG.append({
        "#":          num,
        "step":       name,
        "input":      in_name,
        "in_shape":   shp(in_obj),
        "output":     out_name,
        "out_shape":  shp(out_obj),
        "process":    process,
        "files":      files,
        "calculation": calc,
    })

# ---------------------------------------------------------------------------
@torch.no_grad()
def run_level10(prompt: str):
    STEP_LOG.clear()
    STAGE_LOG.clear()

    # --- 1. tokenize -------------------------------------------------------
    with Stage("1 tokenize", reads="tokenizer.json, tokenizer_config.json, special_tokens_map.json"):
        if TRACE_MODE == "chat":
            messages = [{"role": "user", "content": prompt}]
            enc = tokenizer.apply_chat_template(
                messages, add_generation_prompt=True, return_tensors="pt"
            )
            input_ids = (enc if torch.is_tensor(enc) else enc["input_ids"]).to(DEVICE)
            tok_proc = ("BPE merge + ID map, wrapped with chat/special tokens; "
                        "add_generation_prompt=True -> predicts FIRST token of assistant reply")
        else:  # completion
            input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(DEVICE)
            tok_proc = ("BPE byte-pair merge, map pieces to integer IDs; "
                        "raw completion -> predicts continuation of the text")
    record(1, "tokenize", "text", prompt, "token IDs", input_ids,
           tok_proc,
           "tokenizer.json, tokenizer_config.json, special_tokens_map.json",
           "deterministic table merges + ID lookup; no floating-point math")
    N_tok = input_ids.shape[1]

    # --- 2. embed (row lookup in {E}) -------------------------------------
    with Stage("2 embed", reads="model.embed_tokens.weight"):
        embeds = model.model.embed_tokens(input_ids)  # [1, N_tok, d]
    record(2, "embed", "token IDs", input_ids, "vectors", embeds,
           "row lookup in {E}: row i is the learned vector for token id i",
           f"{WEIGHTS_FILE}  ->  embed_tokens.weight [{vocab_size} x {d}]",
           "out[b,t] = E[ ids[b,t] ]  (indexing, NOT a matmul)")

    # --- 3. LLM core (OPAQUE: rotary + 16 layers + final RMSNorm) ---------
    with Stage("3 LLM core (opaque)", reads="all layer weights + model.norm.weight"):
        core = model.model(inputs_embeds=embeds, use_cache=False)
        hidden = core.last_hidden_state                # [1, N_tok, d], final-normed
    record(3, "LLM core (opaque)", "vectors", embeds, "vectors", hidden,
           "BLACK BOX: rotary pos emb -> 16 decoder layers (attn+FFN, residual, pre-norm) -> final RMSNorm",
           f"config.json (shapes) + {WEIGHTS_FILE} (16 layers + model.norm.weight)",
           "opaque at L10; only invariant asserted: shape in == shape out, now contextualized")

    # --- 4. take last hidden, then LM head (slice BEFORE projection) ------
    #     production-faithful: only the last position predicts the next token,
    #     so we avoid the wasteful [N_tok x vocab_size] matmul.
    with Stage("4 LM head (last pos only)", reads="lm_head.weight (tied to embed_tokens.weight)"):
        last_hidden = hidden[:, -1:, :]                # [1, 1, d]
        last_logits = model.lm_head(last_hidden)[0, -1, :]   # [vocab_size] -> true 1D
    record(4, "LM head (last pos)", "vectors", last_hidden, "logits (last pos)", last_logits,
           f"linear projection onto vocab with {{H}} (TIED to {{E}}: {TIED}); "
           f"slice last position BEFORE matmul (avoids N_tok x vocab_size work)",
           f"{WEIGHTS_FILE}  ->  lm_head.weight [{vocab_size} x {d}] (same tensor as embed)",
           "logits = hidden @ H^T ; H stored as [vocab_size x d], so transpose; "
           "each elem = dot(d-vec hidden, d-vec vocab row)")

    # --- 5. softmax (fp32 for stability; monotonic -> no-op before argmax) -
    with Stage("5 softmax", reads="(none)"):
        probs = torch.softmax(last_logits.float(), dim=-1)   # [vocab_size]
    record(5, "softmax", "logits (last pos)", last_logits, "probabilities", probs,
           "softmax over vocab axis (cast to fp32 first for stability); "
           "NOTE: monotonic, so argmax(softmax(z)) == argmax(z) -> vestigial under greedy",
           "(none)",
           "p_i = exp(z_i - max z) / sum_j exp(z_j - max z) ; sums to 1")

    # --- 6. pick = argmax -------------------------------------------------
    with Stage("6 pick (argmax)", reads="(none)"):
        next_id = torch.argmax(probs, dim=-1)          # scalar
    record(6, "pick (argmax)", "probabilities", probs, "token ID", next_id,
           "greedy: index of the maximum probability",
           "(none)",
           "id = argmax_i p_i ; deterministic")

    # --- 7. decode --------------------------------------------------------
    with Stage("7 decode", reads="tokenizer.json"):
        text_piece = tokenizer.decode(next_id.tolist())
    record(7, "decode", "token ID", next_id, "text piece", text_piece,
           "inverse vocab lookup id -> byte fragment, byte-level BPE reassembly "
           "(leading space is expected, not a bug)",
           "tokenizer.json (vocab + merges)",
           "table lookup + byte decode; no floating-point math")

    return next_id.item(), text_piece

# ---------------------------------------------------------------------------
if __name__ == "__main__":
    tok_id, piece = run_level10("The capital of France is")

    df_steps = pd.DataFrame(STEP_LOG).set_index("#")
    pd.set_option("display.max_colwidth", None)
    pd.set_option("display.width", None)

    print(f"\nsizes: d={d} d_ff={d_ff} n_h={n_h} n_kv={n_kv} d_h={d_h} "
          f"N={N} vocab={vocab_size}  tied_embeddings={TIED}  mode={TRACE_MODE}\n")
    print(df_steps[["step", "input", "in_shape", "output", "out_shape"]].to_string(), "\n")
    print(df_steps[["step", "process"]].to_string(), "\n")
    print(df_steps[["step", "files"]].to_string(), "\n")
    print(df_steps[["step", "calculation"]].to_string(), "\n")
    print(f"predicted token id = {tok_id!r}   text = {piece!r}")


# ---------------------------------------------------------------------------
# Merged timing/RAM table. Works in Jupyter (display) and plain python (print).
# ---------------------------------------------------------------------------
steps = pd.DataFrame(STEP_LOG)

stage = pd.DataFrame(STAGE_LOG)
stage["#"] = stage["stage"].str.extract(r"^(\d+)\s")
stage = stage.dropna(subset=["#"]).astype({"#": int})
timing = (stage[["#", "time_s", "ram_delta_MB"]]
          .drop_duplicates(subset="#", keep="last"))

df = (steps.merge(timing, on="#", how="left").set_index("#")
      [["step", "input", "in_shape", "output", "out_shape",
        "process", "files", "calculation", "time_s", "ram_delta_MB"]])
assert df.index.is_unique, "index still non-unique -> check STEP_LOG/STAGE_LOG"

try:
    display(
        df.style
          .set_properties(**{"text-align": "left", "vertical-align": "top"})
          .set_properties(subset=["process", "files", "calculation"],
                          **{"white-space": "pre-wrap", "max-width": "420px"})
          .set_table_styles([
              {"selector": "th", "props": [("text-align", "left"),
                                           ("vertical-align", "top"), ("padding", "6px")]},
              {"selector": "td", "props": [("padding", "6px"),
                                           ("border", "1px solid #ddd")]},
          ])
          .apply(lambda r: ["background-color:#3a3a3a;color:#f0f0f0"
                            if r.name == 3 else "" for _ in r], axis=1)
    )
except NameError:
    print(df.to_string())

In [1]:
import os, time
import torch
import psutil
import pandas as pd

# Device/dtype: fp16 on GPU; fp32 on CPU. fp16 matmul on CPU is slow and some
# kernels are unsupported, so do NOT force fp16 without a GPU. (bf16 also OK on CPU.)
if torch.cuda.is_available():
    DEVICE, DTYPE = "cuda", torch.float16
else:
    DEVICE, DTYPE = "cpu", torch.float32

PROC = psutil.Process(os.getpid())
def ram_mb(): return PROC.memory_info().rss / (1024 ** 2)   # RSS: includes shared pages -> deltas are noisy

def fmt_size(b):
    if b < 1024:      return f"{b} B"
    if b < 1024**2:   return f"{b/1024:.2f} KB"
    if b < 1024**3:   return f"{b/1024**2:.2f} MB"
    return f"{b/1024**3:.2f} GB"

STAGE_LOG = []

class Stage:
    def __init__(self, name, reads=""):
        self.name, self.reads = name, reads
    def __enter__(self):
        self.ram_start = ram_mb()
        self.t_start = time.time()
        return self
    def __exit__(self, *a):
        dt = time.time() - self.t_start
        ram_end = ram_mb()
        STAGE_LOG.append({
            "stage": self.name, "reads": self.reads,
            "ram_before_MB": round(self.ram_start, 1),
            "ram_after_MB":  round(ram_end, 1),
            "ram_delta_MB":  round(ram_end - self.ram_start, 1),
            "time_s":        round(dt, 2),
        })

MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"

# meta-llama/* is a GATED repo. Before this runs you MUST (1) accept the license
# on the model's HF page and (2) authenticate:  huggingface-cli login
# or set HF_TOKEN=hf_...  Otherwise snapshot_download raises 401/403.
# For a truly offline run: pre-populate the cache, then set HF_HUB_OFFLINE=1.
from huggingface_hub import snapshot_download, scan_cache_dir

def cache_info(model_id):
    try:
        for r in scan_cache_dir().repos:
            if r.repo_id == model_id:
                return True, r.size_on_disk
    except Exception:
        pass
    return False, 0

def categorize(name):
    n = name.lower()
    if n.endswith((".safetensors", ".bin")):                return "weights"
    if n in ("config.json", "generation_config.json"):      return "model-config"
    if "tokenizer" in n or n == "special_tokens_map.json":  return "tokenizer"
    if n.endswith((".md", ".txt")):                         return "metadata/license"
    return "other"

cached_before, size_before = cache_info(MODEL_ID)

with Stage("snapshot_download", reads="huggingface hub"):
    local_path = snapshot_download(repo_id=MODEL_ID)

rows = []
for f in os.listdir(local_path):
    full = os.path.join(local_path, f)
    if os.path.isfile(full) or os.path.islink(full):
        s = os.path.getsize(full)
        rows.append({"filename": f, "category": categorize(f),
                     "size_bytes": s, "size": fmt_size(s)})
df_files = pd.DataFrame(rows).sort_values("size_bytes").reset_index(drop=True)
df_files.index = range(1, len(df_files) + 1)
df_files.index.name = "#"

from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM

ram_baseline = ram_mb()

with Stage("load tokenizer", reads="tokenizer.json, tokenizer_config.json, special_tokens_map.json"):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

with Stage("load model config", reads="config.json"):
    config = AutoConfig.from_pretrained(MODEL_ID)

with Stage("load model weights", reads="*.safetensors"):
    # attn_implementation="eager" makes the manual attention math bit-match HF.
    # If transformers warns/errors on torch_dtype, rename it to dtype=.
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=DTYPE, attn_implementation="eager"
    ).to(DEVICE)                 # from_pretrained loads on CPU by default; move explicitly
    model.eval()

total_params = sum(p.numel() for p in model.parameters())
param_bytes  = sum(p.numel() * p.element_size() for p in model.parameters())
print(f"device={DEVICE} dtype={DTYPE} params={total_params:,} weights={fmt_size(param_bytes)}")

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

device=cpu dtype=torch.float32 params=1,235,814,400 weights=4.60 GB


In [2]:
# ---------------------------------------------------------------------------
# Level 49 — complete one-inference-one-token reference. Every step exposed.
# Attention math mirrors transformers' eager_attention_forward EXACTLY
# (score matmul in model dtype, softmax upcast to fp32, finfo.min mask),
# so verify() matches HF bit-for-bit. LM head runs over ALL positions and
# "take last row" is its own step, matching the documented table.
# Requires: attn_implementation="eager" + transformers >= 4.43 (model-level
# rotary_emb with (x, position_ids) signature). Reuses Cell-1 objects.
# ---------------------------------------------------------------------------
import math
import torch
import torch.nn.functional as F
import pandas as pd
from transformers.models.llama.modeling_llama import rotate_half, repeat_kv

d          = config.hidden_size
d_ff       = config.intermediate_size
n_h        = config.num_attention_heads
n_kv       = config.num_key_value_heads
d_h        = d // n_h
n_rep      = n_h // n_kv
N          = config.num_hidden_layers
vocab_size = config.vocab_size
max_pos    = config.max_position_embeddings

DEVICE = next(model.parameters()).device
DTYPE  = next(model.parameters()).dtype
MIN    = torch.finfo(DTYPE).min                       # HF uses this, not -inf
WEIGHTS_FILE = "model.safetensors (resident after load; not re-read per step)"
TIED = model.lm_head.weight.data_ptr() == model.model.embed_tokens.weight.data_ptr()

TRACE_MODE = "completion"   # "completion" | "chat"

STEP_LOG = []

def shp(x):
    if isinstance(x, torch.Tensor):
        return "scalar" if x.dim() == 0 else "[" + " x ".join(map(str, x.shape)) + "]"
    if isinstance(x, str):
        return "string"
    return type(x).__name__

def record(num, name, in_name, in_obj, out_name, out_obj, process, files, calc):
    STEP_LOG.append({
        "#": num, "step": name,
        "input": in_name, "in_shape": shp(in_obj),
        "output": out_name, "out_shape": shp(out_obj),
        "process": process, "files": files, "calculation": calc,
    })

def get_rope(hidden, position_ids):
    rot = getattr(model.model, "rotary_emb", None)        # >=4.43
    if rot is None:
        rot = model.model.layers[0].self_attn.rotary_emb  # some 4.4x builds
    return rot(hidden, position_ids)                       # -> (cos, sin) each [1, N_tok, d_h]

def apply_rope_single(t, cos_b, sin_b):
    return (t * cos_b) + (rotate_half(t) * sin_b)

@torch.no_grad()
def run_level49(prompt: str):
    STEP_LOG.clear()
    STAGE_LOG.clear()

    # --- 1. tokenize -------------------------------------------------------
    with Stage("1 tokenize", reads="tokenizer.json, tokenizer_config.json, special_tokens_map.json"):
        if TRACE_MODE == "chat":
            messages = [{"role": "user", "content": prompt}]
            enc = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt")
            input_ids = (enc if torch.is_tensor(enc) else enc["input_ids"]).to(DEVICE)
            tok_proc = "BPE merge + ID map, chat/special tokens; predicts FIRST assistant token"
        else:
            input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(DEVICE)
            tok_proc = "BPE byte-pair merge + ID map; raw completion -> predicts continuation"
    record(1, "tokenize", "text", prompt, "token IDs", input_ids,
           tok_proc, "tokenizer.json, tokenizer_config.json, special_tokens_map.json",
           "deterministic table merges + ID lookup; no floating-point math")
    N_tok = input_ids.shape[1]

    # --- 2. embed ----------------------------------------------------------
    with Stage("2 embed", reads="model.embed_tokens.weight"):
        hidden = model.model.embed_tokens(input_ids)       # [1, N_tok, d]
    record(2, "embed", "token IDs", input_ids, "vectors", hidden,
           "row lookup in {E}: row i is the learned vector for token id i",
           f"{WEIGHTS_FILE} -> embed_tokens.weight [{vocab_size} x {d}]",
           "out[b,t] = E[ ids[b,t] ]  (indexing, NOT a matmul)")

    # shared scaffolding (built once, threaded into every block's attention)
    position_ids = torch.arange(N_tok, device=DEVICE).unsqueeze(0)
    cos, sin = get_rope(hidden, position_ids)
    cos_b, sin_b = cos.unsqueeze(1), sin.unsqueeze(1)      # [1,1,N_tok,d_h]
    causal_mask = torch.full((N_tok, N_tok), MIN, dtype=DTYPE, device=DEVICE).triu(1)
    scale = 1.0 / math.sqrt(d_h)

    step = 3
    for i, layer in enumerate(model.model.layers):
        sa, mlp = layer.self_attn, layer.mlp

        # ===================== ATTENTION sub-block (3.i.a) =====================
        with Stage(f"{step} block{i} save x", reads="(none)"):
            x_held = hidden
        record(step, f"3.{i}.a.save : save x", "vectors (x)", hidden, "x held", x_held,
               "identity: hold pre-norm x for residual", "(none)", "plumbing only"); step += 1

        with Stage(f"{step} block{i} pre-attn norm", reads=f"layer[{i}].input_layernorm.weight"):
            x_normed = layer.input_layernorm(hidden)
        record(step, f"3.{i}.a.norm : pre-attn norm", "x", hidden, "x_normed", x_normed,
               "RMSNorm with {attn_i.norm}",
               f"{WEIGHTS_FILE} -> input_layernorm.weight [{d}]",
               "y = x / sqrt(mean(x^2)+eps) * weight"); step += 1

        with Stage(f"{step} block{i} produce Q", reads=f"layer[{i}].self_attn.q_proj.weight"):
            Q = sa.q_proj(x_normed)
        record(step, f"3.{i}.a.q : produce Q", "x_normed", x_normed, "Q flat", Q,
               "matmul x_normed @ W_Q^T", f"{WEIGHTS_FILE} -> q_proj.weight [{n_h*d_h} x {d}]",
               "Q = x_normed @ W_Q"); step += 1

        with Stage(f"{step} block{i} split Q heads", reads="(none)"):
            q = Q.view(1, N_tok, n_h, d_h).transpose(1, 2)
        record(step, f"3.{i}.a.q_split : split Q into heads", "Q flat", Q, "Q per-head", q,
               f"reshape d={d} into n_h={n_h} heads of d_h={d_h}", "(none)",
               "[N_tok, n_h*d_h] -> [n_h, N_tok, d_h]"); step += 1

        with Stage(f"{step} block{i} produce K", reads=f"layer[{i}].self_attn.k_proj.weight"):
            K = sa.k_proj(x_normed)
        record(step, f"3.{i}.a.k : produce K", "x_normed", x_normed, "K flat", K,
               "matmul x_normed @ W_K^T (GQA: n_kv heads)",
               f"{WEIGHTS_FILE} -> k_proj.weight [{n_kv*d_h} x {d}]", "K = x_normed @ W_K"); step += 1

        with Stage(f"{step} block{i} split K heads", reads="(none)"):
            k = K.view(1, N_tok, n_kv, d_h).transpose(1, 2)
        record(step, f"3.{i}.a.k_split : split K into heads", "K flat", K, "K per-head", k,
               f"reshape into n_kv={n_kv} heads of d_h={d_h}", "(none)",
               "[N_tok, n_kv*d_h] -> [n_kv, N_tok, d_h]"); step += 1

        with Stage(f"{step} block{i} produce V", reads=f"layer[{i}].self_attn.v_proj.weight"):
            V = sa.v_proj(x_normed)
        record(step, f"3.{i}.a.v : produce V", "x_normed", x_normed, "V flat", V,
               "matmul x_normed @ W_V^T (GQA: n_kv heads); V never rotated",
               f"{WEIGHTS_FILE} -> v_proj.weight [{n_kv*d_h} x {d}]", "V = x_normed @ W_V"); step += 1

        with Stage(f"{step} block{i} split V heads", reads="(none)"):
            v = V.view(1, N_tok, n_kv, d_h).transpose(1, 2)
        record(step, f"3.{i}.a.v_split : split V into heads", "V flat", V, "V per-head", v,
               f"reshape into n_kv={n_kv} heads of d_h={d_h}", "(none)",
               "[N_tok, n_kv*d_h] -> [n_kv, N_tok, d_h]"); step += 1

        with Stage(f"{step} block{i} rope_q", reads="{RoPE} sin/cos"):
            q_rot = apply_rope_single(q, cos_b, sin_b)
        record(step, f"3.{i}.a.rope_q : rotate Q", "Q per-head", q, "Q_rot", q_rot,
               "RoPE: rotate each Q head by position-dependent angle",
               f"{{RoPE : {max_pos} x {d_h}}} (pre-built)",
               "q_rot = q*cos + rotate_half(q)*sin"); step += 1

        with Stage(f"{step} block{i} rope_k", reads="{RoPE} sin/cos"):
            k_rot = apply_rope_single(k, cos_b, sin_b)
        record(step, f"3.{i}.a.rope_k : rotate K", "K per-head", k, "K_rot", k_rot,
               "RoPE: rotate each K head; V untouched -> scores depend on relative position",
               f"{{RoPE : {max_pos} x {d_h}}} (pre-built)",
               "k_rot = k*cos + rotate_half(k)*sin"); step += 1

        k_g = repeat_kv(k_rot, n_rep)                      # GQA broadcast n_kv -> n_h
        v_g = repeat_kv(v, n_rep)

        with Stage(f"{step} block{i} op.scores", reads="(none)"):
            raw_scores = torch.matmul(q_rot, k_g.transpose(2, 3))   # MODEL dtype, like HF eager
        record(step, f"3.{i}.a.op.scores : Q.K^T", "Q_rot, K_rot", q_rot, "raw scores", raw_scores,
               f"per-head Q.K^T; K/V group-repeated n_rep={n_rep}x (n_kv->n_h)", "(none)",
               "scores[h,i,j] = dot(q_rot[h,i,:], k_rot[group(h),j,:])"); step += 1

        with Stage(f"{step} block{i} op.scale", reads="(none)"):
            scaled = raw_scores * scale
        record(step, f"3.{i}.a.op.scale : scale 1/sqrt(d_h)", "raw scores", raw_scores,
               "scaled scores", scaled, f"multiply by 1/sqrt(d_h)={scale:.5f}", "(none)",
               "scaled = raw / sqrt(d_h)"); step += 1

        with Stage(f"{step} block{i} op.mask", reads="{causal_mask}"):
            masked = scaled + causal_mask
        record(step, f"3.{i}.a.op.mask : causal mask", "scaled scores", scaled, "masked scores", masked,
               "add finfo.min to future positions (j > i)", "{causal_mask : N_tok x N_tok}",
               "masked[h,i,j] = scaled + (0 if j<=i else finfo.min)"); step += 1

        with Stage(f"{step} block{i} op.softmax", reads="(none)"):
            attn_w = torch.softmax(masked, dim=-1, dtype=torch.float32).to(DTYPE)  # HF eager
        record(step, f"3.{i}.a.op.smax : softmax over keys", "masked scores", masked,
               "attn weights", attn_w, "softmax along KEY axis (upcast fp32, like HF)", "(none)",
               "w[h,i,:] = softmax_j(masked[h,i,j])"); step += 1

        with Stage(f"{step} block{i} op.weighted", reads="(none)"):
            per_head = torch.matmul(attn_w, v_g)
        record(step, f"3.{i}.a.op.weighted : weighted sum with V", "attn weights, V", attn_w,
               "per-head results", per_head, "weighted sum of V rows per query per head", "(none)",
               "ctx[h,i,:] = sum_j w[h,i,j] * v[group(h),j,:]"); step += 1

        with Stage(f"{step} block{i} concat heads", reads="(none)"):
            merged = per_head.transpose(1, 2).contiguous().view(1, N_tok, d)
        record(step, f"3.{i}.a.concat : concat heads", "per-head results", per_head,
               "merged result", merged, f"concat n_h={n_h} heads into d={d} per token", "(none)",
               "[n_h, N_tok, d_h] -> [N_tok, d]"); step += 1

        with Stage(f"{step} block{i} output projection", reads=f"layer[{i}].self_attn.o_proj.weight"):
            attn_delta = sa.o_proj(merged)
        record(step, f"3.{i}.a.proj : output projection", "merged result", merged, "attn delta", attn_delta,
               "matmul merged @ W_O^T; W_O mixes information across heads",
               f"{WEIGHTS_FILE} -> o_proj.weight [{d} x {d}]", "attn_delta = merged @ W_O"); step += 1

        with Stage(f"{step} block{i} add residual (attn)", reads="(none)"):
            hidden = x_held + attn_delta
        record(step, f"3.{i}.a.add : add residual", "x held + attn delta", attn_delta, "vectors (y)", hidden,
               "elementwise residual add (pre-norm arch)", "(none)", "y = x_held + attn_delta"); step += 1

        # ======================== FFN sub-block (3.i.b) ========================
        with Stage(f"{step} block{i} save y", reads="(none)"):
            y_held = hidden
        record(step, f"3.{i}.b.save : save y", "vectors (y)", hidden, "y held", y_held,
               "identity: hold post-attention y for the FFN residual", "(none)", "plumbing only"); step += 1

        with Stage(f"{step} block{i} pre-FFN norm", reads=f"layer[{i}].post_attention_layernorm.weight"):
            y_normed = layer.post_attention_layernorm(hidden)
        record(step, f"3.{i}.b.norm : pre-FFN norm", "y", hidden, "y_normed", y_normed,
               "RMSNorm with {ffn_i.norm}",
               f"{WEIGHTS_FILE} -> post_attention_layernorm.weight [{d}]",
               "y_normed = y / sqrt(mean(y^2)+eps) * weight"); step += 1

        with Stage(f"{step} block{i} gate projection", reads=f"layer[{i}].mlp.gate_proj.weight"):
            gate = mlp.gate_proj(y_normed)
        record(step, f"3.{i}.b.gate : gate projection", "y_normed", y_normed, "gate", gate,
               "matmul y_normed @ W_gate^T; lifts d -> d_ff (parallel to up)",
               f"{WEIGHTS_FILE} -> gate_proj.weight [{d_ff} x {d}]", "gate = y_normed @ W_gate"); step += 1

        with Stage(f"{step} block{i} SiLU", reads="(none)"):
            gate_act = F.silu(gate)
        record(step, f"3.{i}.b.silu : SiLU", "gate", gate, "gate_act", gate_act,
               "elementwise SiLU on gate only; up is NOT activated", "(none)",
               "silu(x) = x * sigmoid(x)"); step += 1

        with Stage(f"{step} block{i} up projection", reads=f"layer[{i}].mlp.up_proj.weight"):
            up = mlp.up_proj(y_normed)
        record(step, f"3.{i}.b.up : up projection", "y_normed", y_normed, "up", up,
               "matmul y_normed @ W_up^T; lifts d -> d_ff (parallel to gate)",
               f"{WEIGHTS_FILE} -> up_proj.weight [{d_ff} x {d}]", "up = y_normed @ W_up"); step += 1

        with Stage(f"{step} block{i} combine (gate_act * up)", reads="(none)"):
            ffn_hidden = gate_act * up
        record(step, f"3.{i}.b.combine : elementwise multiply", "gate_act, up", gate_act, "hidden", ffn_hidden,
               "SwiGLU gating: gate_act modulates up dimension-by-dimension", "(none)",
               "hidden = gate_act (.) up  (Hadamard product)"); step += 1

        with Stage(f"{step} block{i} down projection", reads=f"layer[{i}].mlp.down_proj.weight"):
            ffn_delta = mlp.down_proj(ffn_hidden)
        record(step, f"3.{i}.b.down : down projection", "hidden", ffn_hidden, "ffn delta", ffn_delta,
               "matmul hidden @ W_down^T; projects d_ff -> d",
               f"{WEIGHTS_FILE} -> down_proj.weight [{d} x {d_ff}]", "ffn_delta = hidden @ W_down"); step += 1

        with Stage(f"{step} block{i} add residual (FFN)", reads="(none)"):
            hidden = y_held + ffn_delta
        record(step, f"3.{i}.b.add : add residual", "y held + ffn delta", ffn_delta, "vectors", hidden,
               "elementwise residual add (pre-norm arch)", "(none)", "h = y_held + ffn_delta"); step += 1

    # --- final norm --------------------------------------------------------
    with Stage(f"{step} final norm", reads="model.norm.weight"):
        hidden = model.model.norm(hidden)
    record(step, "final norm", "vectors", hidden, "vectors", hidden,
           "RMSNorm with {final_norm}", f"{WEIGHTS_FILE} -> model.norm.weight [{d}]",
           "y = x / sqrt(mean(x^2)+eps) * weight"); step += 1

    # --- LM head over ALL positions (doc-faithful; matches HF default forward)
    with Stage(f"{step} LM head", reads="lm_head.weight (tied to embed_tokens)"):
        logits_all = model.lm_head(hidden)                 # [1, N_tok, vocab_size]
    record(step, "LM head", "vectors", hidden, "logits (all pos)", logits_all,
           f"linear projection onto vocab with {{H}} (TIED: {TIED})",
           f"{WEIGHTS_FILE} -> lm_head.weight [{vocab_size} x {d}] (same tensor as embed)",
           "logits = hidden @ H^T ; only step that changes per-token dim"); step += 1

    with Stage(f"{step} take last row", reads="(none)"):
        last_logits = logits_all[0, -1, :]                 # [vocab_size]
    record(step, "take last row", "logits (all pos)", logits_all, "logits (last pos)", last_logits,
           "slice sequence axis at -1; only last position predicts the next token", "(none)",
           "out = logits[0, -1, :]"); step += 1

    with Stage(f"{step} softmax", reads="(none)"):
        probs = torch.softmax(last_logits.float(), dim=-1) # [vocab_size]
    record(step, "softmax", "logits (last pos)", last_logits, "probabilities", probs,
           "softmax over vocab (fp32); monotonic -> vestigial under greedy argmax", "(none)",
           "p_i = exp(z_i - max z)/sum_j exp(z_j - max z)"); step += 1

    with Stage(f"{step} pick (argmax)", reads="(none)"):
        next_id = torch.argmax(probs, dim=-1)
    record(step, "pick (argmax)", "probabilities", probs, "token ID", next_id,
           "greedy: index of maximum probability", "(none)", "id = argmax_i p_i"); step += 1

    with Stage(f"{step} decode", reads="tokenizer.json"):
        text_piece = tokenizer.decode(next_id.tolist())
    record(step, "decode", "token ID", next_id, "text piece", text_piece,
           "inverse vocab lookup + byte-level BPE reassembly (leading space expected)",
           "tokenizer.json (vocab + merges)", "table lookup + byte decode; no FP math")

    return next_id.item(), text_piece, last_logits

# ---------------------------------------------------------------------------
@torch.no_grad()
def verify(prompt: str):
    if TRACE_MODE == "chat":
        messages = [{"role": "user", "content": prompt}]
        enc = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt")
        ids = (enc if torch.is_tensor(enc) else enc["input_ids"]).to(DEVICE)
    else:
        ids = tokenizer(prompt, return_tensors="pt").input_ids.to(DEVICE)
    ref = model(ids).logits[0, -1, :].float()
    my_id, _, my_logits = run_level49(prompt)
    ref_id = torch.argmax(ref).item()
    max_abs = (ref - my_logits.float()).abs().max().item()
    return ref_id, my_id, (ref_id == my_id), max_abs

if __name__ == "__main__":
    tok_id, piece, _ = run_level49("The capital of France is")
    df = pd.DataFrame(STEP_LOG).set_index("#")
    pd.set_option("display.max_colwidth", None); pd.set_option("display.width", None)
    print(f"\nsizes: d={d} d_ff={d_ff} n_h={n_h} n_kv={n_kv} d_h={d_h} n_rep={n_rep} "
          f"N={N} vocab={vocab_size} tied={TIED} mode={TRACE_MODE}  total_steps={len(df)}\n")
    print(df[["step", "input", "in_shape", "output", "out_shape"]].head(40).to_string(), "\n")
    print(f"predicted token id = {tok_id!r}   text = {piece!r}")

    ref_id, my_id, ok, max_abs = verify("The capital of France is")
    print(f"\nverify: reference_argmax={ref_id}  manual_argmax={my_id}  match={ok}  "
          f"max|Δlogit|={max_abs:.3e}")

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



sizes: d=2048 d_ff=8192 n_h=32 n_kv=8 d_h=64 n_rep=4 N=16 vocab=128256 tied=True mode=completion  total_steps=424

                                       step                input           in_shape            output          out_shape
#                                                                                                                       
1                                  tokenize                 text             string         token IDs            [1 x 6]
2                                     embed            token IDs            [1 x 6]           vectors     [1 x 6 x 2048]
3                       3.0.a.save : save x          vectors (x)     [1 x 6 x 2048]            x held     [1 x 6 x 2048]
4                3.0.a.norm : pre-attn norm                    x     [1 x 6 x 2048]          x_normed     [1 x 6 x 2048]
5                       3.0.a.q : produce Q             x_normed     [1 x 6 x 2048]            Q flat     [1 x 6 x 2048]
6        3.0.a.q_split : split Q into

In [3]:
N = config.num_hidden_layers
print("N (layers / loop count):", N)
print("documented step-types:", 34)
print("executed steps per inference:", 2 + 26 * N + 6)
print("recorded STEP_LOG rows:", len(STEP_LOG))   # after one run_level49(...) call

N (layers / loop count): 16
documented step-types: 34
executed steps per inference: 424
recorded STEP_LOG rows: 424


In [4]:
import pandas as pd
steps = pd.DataFrame(STEP_LOG)
stage = pd.DataFrame(STAGE_LOG)
stage["#"] = stage["stage"].str.extract(r"^(\d+)\s").astype("Int64")
stage = stage.dropna(subset=["#"]).astype({"#": int})

df = steps.merge(stage[["#", "time_s", "ram_delta_MB"]], on="#", how="left")

# strip the per-layer index ("3.0.a.q" -> "a.q") to aggregate across all 16 layers
df["kind"] = df["step"].str.replace(r"^3\.\d+\.", "3.i.", regex=True)
by_kind = (df.groupby("kind")
             .agg(count=("kind", "size"), total_time_s=("time_s", "sum"))
             .sort_values("total_time_s", ascending=False))
print(by_kind.to_string())

                                         count  total_time_s
kind                                                        
3.i.b.gate : gate projection                16          0.48
LM head                                      1          0.45
3.i.b.up : up projection                    16          0.44
3.i.b.down : down projection                16          0.41
3.i.a.q : produce Q                         16          0.16
3.i.a.proj : output projection              16          0.16
3.i.a.concat : concat heads                 16          0.00
3.i.a.add : add residual                    16          0.00
3.i.a.k : produce K                         16          0.00
3.i.a.k_split : split K into heads          16          0.00
3.i.a.op.weighted : weighted sum with V     16          0.00
3.i.a.op.smax : softmax over keys           16          0.00
3.i.a.q_split : split Q into heads          16          0.00
3.i.a.rope_k : rotate K                     16          0.00
3.i.a.op.scale : scale 1

In [7]:
# ---------------------------------------------------------------------------
# Level 49 — complete one-inference-one-token reference, LOOP-EXPLICIT.
# Identical math to the flat version; only the STEP_LOG labeling changes so the
# 16x transformer-block loop is visible in the records.
#   ID scheme:  pre.NN  (before the loop)
#               L{ii}.a.*  / L{ii}.b.*   (inside the loop, ii = layer index)
#               post.NN (after the loop)
#   Extra columns: loop ('transformer_block' inside, '—' outside)
#                  iter (0..N-1 inside, '—' outside)
# Requires attn_implementation="eager" + transformers >= 4.43. Reuses Cell-1.
# ---------------------------------------------------------------------------
import math
import torch
import torch.nn.functional as F
import pandas as pd
from transformers.models.llama.modeling_llama import rotate_half, repeat_kv

d          = config.hidden_size
d_ff       = config.intermediate_size
n_h        = config.num_attention_heads
n_kv       = config.num_key_value_heads
d_h        = d // n_h
n_rep      = n_h // n_kv
N          = config.num_hidden_layers
vocab_size = config.vocab_size
max_pos    = config.max_position_embeddings

DEVICE = next(model.parameters()).device
DTYPE  = next(model.parameters()).dtype
MIN    = torch.finfo(DTYPE).min
WEIGHTS_FILE = "model.safetensors (resident after load; not re-read per step)"
TIED = model.lm_head.weight.data_ptr() == model.model.embed_tokens.weight.data_ptr()

TRACE_MODE = "completion"   # "completion" | "chat"

STEP_LOG = []

def shp(x):
    if isinstance(x, torch.Tensor):
        return "scalar" if x.dim() == 0 else "[" + " x ".join(map(str, x.shape)) + "]"
    if isinstance(x, str):
        return "string"
    return type(x).__name__

def record(step_id, loop, it, name, in_name, in_obj, out_name, out_obj, process, files, calc):
    STEP_LOG.append({
        "id":        step_id,                 # hierarchical: pre.NN / L{ii}.x / post.NN
        "loop":      loop,                    # 'transformer_block' or '—'
        "iter":      it,                      # 0..N-1 inside the loop, else '—'
        "step":      name,
        "input":     in_name, "in_shape": shp(in_obj),
        "output":    out_name, "out_shape": shp(out_obj),
        "process":   process, "files": files, "calculation": calc,
    })

def get_rope(hidden, position_ids):
    rot = getattr(model.model, "rotary_emb", None)
    if rot is None:
        rot = model.model.layers[0].self_attn.rotary_emb
    return rot(hidden, position_ids)

def apply_rope_single(t, cos_b, sin_b):
    return (t * cos_b) + (rotate_half(t) * sin_b)

@torch.no_grad()
def run_level49(prompt: str):
    STEP_LOG.clear()
    STAGE_LOG.clear()

    OUT = "—"   # placeholder for loop/iter on outer (non-looped) steps

    # ============================ PRE-LOOP ============================
    # --- pre.01 tokenize ---
    with Stage("pre.01 tokenize", reads="tokenizer.json, tokenizer_config.json, special_tokens_map.json"):
        if TRACE_MODE == "chat":
            messages = [{"role": "user", "content": prompt}]
            enc = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt")
            input_ids = (enc if torch.is_tensor(enc) else enc["input_ids"]).to(DEVICE)
            tok_proc = "BPE merge + ID map, chat/special tokens; predicts FIRST assistant token"
        else:
            input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(DEVICE)
            tok_proc = "BPE byte-pair merge + ID map; raw completion -> predicts continuation"
    record("pre.01", OUT, OUT, "tokenize", "text", prompt, "token IDs", input_ids,
           tok_proc, "tokenizer.json, tokenizer_config.json, special_tokens_map.json",
           "deterministic table merges + ID lookup; no floating-point math")
    N_tok = input_ids.shape[1]

    # --- pre.02 embed ---
    with Stage("pre.02 embed", reads="model.embed_tokens.weight"):
        hidden = model.model.embed_tokens(input_ids)       # [1, N_tok, d]
    record("pre.02", OUT, OUT, "embed", "token IDs", input_ids, "vectors", hidden,
           "row lookup in {E}: row i is the learned vector for token id i",
           f"{WEIGHTS_FILE} -> embed_tokens.weight [{vocab_size} x {d}]",
           "out[b,t] = E[ ids[b,t] ]  (indexing, NOT a matmul)")

    # shared scaffolding (built once, threaded into every block's attention)
    position_ids = torch.arange(N_tok, device=DEVICE).unsqueeze(0)
    cos, sin = get_rope(hidden, position_ids)
    cos_b, sin_b = cos.unsqueeze(1), sin.unsqueeze(1)
    causal_mask = torch.full((N_tok, N_tok), MIN, dtype=DTYPE, device=DEVICE).triu(1)
    scale = 1.0 / math.sqrt(d_h)

    # ===================== LOOP: transformer_block x N =====================
    # Each iteration i runs the 26 in-block steps. IDs carry the layer index
    # (L00..L{N-1}); the 'iter' column makes per-iteration grouping trivial.
    for i, layer in enumerate(model.model.layers):
        sa, mlp = layer.self_attn, layer.mlp
        ii  = f"L{i:02d}"                      # layer tag, e.g. L00, L07, L15
        LB  = "transformer_block"
        P   = f"{ii}.a"                        # attention sub-block prefix
        Fp  = f"{ii}.b"                        # FFN sub-block prefix

        # ----------------- attention sub-block (a) -----------------
        with Stage(f"{P}.save block{i} save x", reads="(none)"):
            x_held = hidden
        record(f"{P}.save", LB, i, "attn save x", "vectors (x)", hidden, "x held", x_held,
               "identity: hold pre-norm x for residual", "(none)", "plumbing only")

        with Stage(f"{P}.norm block{i} pre-attn norm", reads=f"layer[{i}].input_layernorm.weight"):
            x_normed = layer.input_layernorm(hidden)
        record(f"{P}.norm", LB, i, "pre-attn norm", "x", hidden, "x_normed", x_normed,
               "RMSNorm with {attn_i.norm}", f"{WEIGHTS_FILE} -> input_layernorm.weight [{d}]",
               "y = x / sqrt(mean(x^2)+eps) * weight")

        with Stage(f"{P}.q block{i} produce Q", reads=f"layer[{i}].self_attn.q_proj.weight"):
            Q = sa.q_proj(x_normed)
        record(f"{P}.q", LB, i, "produce Q", "x_normed", x_normed, "Q flat", Q,
               "matmul x_normed @ W_Q^T", f"{WEIGHTS_FILE} -> q_proj.weight [{n_h*d_h} x {d}]",
               "Q = x_normed @ W_Q")

        with Stage(f"{P}.q_split block{i} split Q heads", reads="(none)"):
            q = Q.view(1, N_tok, n_h, d_h).transpose(1, 2)
        record(f"{P}.q_split", LB, i, "split Q into heads", "Q flat", Q, "Q per-head", q,
               f"reshape d={d} into n_h={n_h} heads of d_h={d_h}", "(none)",
               "[N_tok, n_h*d_h] -> [n_h, N_tok, d_h]")

        with Stage(f"{P}.k block{i} produce K", reads=f"layer[{i}].self_attn.k_proj.weight"):
            K = sa.k_proj(x_normed)
        record(f"{P}.k", LB, i, "produce K", "x_normed", x_normed, "K flat", K,
               "matmul x_normed @ W_K^T (GQA: n_kv heads)",
               f"{WEIGHTS_FILE} -> k_proj.weight [{n_kv*d_h} x {d}]", "K = x_normed @ W_K")

        with Stage(f"{P}.k_split block{i} split K heads", reads="(none)"):
            k = K.view(1, N_tok, n_kv, d_h).transpose(1, 2)
        record(f"{P}.k_split", LB, i, "split K into heads", "K flat", K, "K per-head", k,
               f"reshape into n_kv={n_kv} heads of d_h={d_h}", "(none)",
               "[N_tok, n_kv*d_h] -> [n_kv, N_tok, d_h]")

        with Stage(f"{P}.v block{i} produce V", reads=f"layer[{i}].self_attn.v_proj.weight"):
            V = sa.v_proj(x_normed)
        record(f"{P}.v", LB, i, "produce V", "x_normed", x_normed, "V flat", V,
               "matmul x_normed @ W_V^T (GQA: n_kv heads); V never rotated",
               f"{WEIGHTS_FILE} -> v_proj.weight [{n_kv*d_h} x {d}]", "V = x_normed @ W_V")

        with Stage(f"{P}.v_split block{i} split V heads", reads="(none)"):
            v = V.view(1, N_tok, n_kv, d_h).transpose(1, 2)
        record(f"{P}.v_split", LB, i, "split V into heads", "V flat", V, "V per-head", v,
               f"reshape into n_kv={n_kv} heads of d_h={d_h}", "(none)",
               "[N_tok, n_kv*d_h] -> [n_kv, N_tok, d_h]")

        with Stage(f"{P}.rope_q block{i} rope_q", reads="{RoPE} sin/cos"):
            q_rot = apply_rope_single(q, cos_b, sin_b)
        record(f"{P}.rope_q", LB, i, "rotate Q (RoPE)", "Q per-head", q, "Q_rot", q_rot,
               "RoPE: rotate each Q head by position-dependent angle",
               f"{{RoPE : {max_pos} x {d_h}}} (pre-built)", "q_rot = q*cos + rotate_half(q)*sin")

        with Stage(f"{P}.rope_k block{i} rope_k", reads="{RoPE} sin/cos"):
            k_rot = apply_rope_single(k, cos_b, sin_b)
        record(f"{P}.rope_k", LB, i, "rotate K (RoPE)", "K per-head", k, "K_rot", k_rot,
               "RoPE: rotate each K head; V untouched -> relative position",
               f"{{RoPE : {max_pos} x {d_h}}} (pre-built)", "k_rot = k*cos + rotate_half(k)*sin")

        k_g = repeat_kv(k_rot, n_rep)          # GQA broadcast n_kv -> n_h
        v_g = repeat_kv(v, n_rep)

        with Stage(f"{P}.op.scores block{i} op.scores", reads="(none)"):
            raw_scores = torch.matmul(q_rot, k_g.transpose(2, 3))   # model dtype, like HF eager
        record(f"{P}.op.scores", LB, i, "op: Q.K^T scores", "Q_rot, K_rot", q_rot, "raw scores", raw_scores,
               f"per-head Q.K^T; K/V group-repeated n_rep={n_rep}x (n_kv->n_h)", "(none)",
               "scores[h,i,j] = dot(q_rot[h,i,:], k_rot[group(h),j,:])")

        with Stage(f"{P}.op.scale block{i} op.scale", reads="(none)"):
            scaled = raw_scores * scale
        record(f"{P}.op.scale", LB, i, "op: scale 1/sqrt(d_h)", "raw scores", raw_scores,
               "scaled scores", scaled, f"multiply by 1/sqrt(d_h)={scale:.5f}", "(none)",
               "scaled = raw / sqrt(d_h)")

        with Stage(f"{P}.op.mask block{i} op.mask", reads="{causal_mask}"):
            masked = scaled + causal_mask
        record(f"{P}.op.mask", LB, i, "op: causal mask", "scaled scores", scaled, "masked scores", masked,
               "add finfo.min to future positions (j > i)", "{causal_mask : N_tok x N_tok}",
               "masked[h,i,j] = scaled + (0 if j<=i else finfo.min)")

        with Stage(f"{P}.op.smax block{i} op.softmax", reads="(none)"):
            attn_w = torch.softmax(masked, dim=-1, dtype=torch.float32).to(DTYPE)  # HF eager
        record(f"{P}.op.smax", LB, i, "op: softmax over keys", "masked scores", masked,
               "attn weights", attn_w, "softmax along KEY axis (upcast fp32, like HF)", "(none)",
               "w[h,i,:] = softmax_j(masked[h,i,j])")

        with Stage(f"{P}.op.weighted block{i} op.weighted", reads="(none)"):
            per_head = torch.matmul(attn_w, v_g)
        record(f"{P}.op.weighted", LB, i, "op: weighted sum with V", "attn weights, V", attn_w,
               "per-head results", per_head, "weighted sum of V rows per query per head", "(none)",
               "ctx[h,i,:] = sum_j w[h,i,j] * v[group(h),j,:]")

        with Stage(f"{P}.concat block{i} concat heads", reads="(none)"):
            merged = per_head.transpose(1, 2).contiguous().view(1, N_tok, d)
        record(f"{P}.concat", LB, i, "concat heads", "per-head results", per_head,
               "merged result", merged, f"concat n_h={n_h} heads into d={d} per token", "(none)",
               "[n_h, N_tok, d_h] -> [N_tok, d]")

        with Stage(f"{P}.proj block{i} output projection", reads=f"layer[{i}].self_attn.o_proj.weight"):
            attn_delta = sa.o_proj(merged)
        record(f"{P}.proj", LB, i, "output projection", "merged result", merged, "attn delta", attn_delta,
               "matmul merged @ W_O^T; W_O mixes across heads",
               f"{WEIGHTS_FILE} -> o_proj.weight [{d} x {d}]", "attn_delta = merged @ W_O")

        with Stage(f"{P}.add block{i} add residual (attn)", reads="(none)"):
            hidden = x_held + attn_delta
        record(f"{P}.add", LB, i, "add residual (attn)", "x held + attn delta", attn_delta,
               "vectors (y)", hidden, "elementwise residual add (pre-norm arch)", "(none)",
               "y = x_held + attn_delta")

        # ----------------- FFN sub-block (b) -----------------
        with Stage(f"{Fp}.save block{i} save y", reads="(none)"):
            y_held = hidden
        record(f"{Fp}.save", LB, i, "ffn save y", "vectors (y)", hidden, "y held", y_held,
               "identity: hold post-attention y for the FFN residual", "(none)", "plumbing only")

        with Stage(f"{Fp}.norm block{i} pre-FFN norm", reads=f"layer[{i}].post_attention_layernorm.weight"):
            y_normed = layer.post_attention_layernorm(hidden)
        record(f"{Fp}.norm", LB, i, "pre-FFN norm", "y", hidden, "y_normed", y_normed,
               "RMSNorm with {ffn_i.norm}", f"{WEIGHTS_FILE} -> post_attention_layernorm.weight [{d}]",
               "y_normed = y / sqrt(mean(y^2)+eps) * weight")

        with Stage(f"{Fp}.gate block{i} gate projection", reads=f"layer[{i}].mlp.gate_proj.weight"):
            gate = mlp.gate_proj(y_normed)
        record(f"{Fp}.gate", LB, i, "gate projection", "y_normed", y_normed, "gate", gate,
               "matmul y_normed @ W_gate^T; lifts d -> d_ff (parallel to up)",
               f"{WEIGHTS_FILE} -> gate_proj.weight [{d_ff} x {d}]", "gate = y_normed @ W_gate")

        with Stage(f"{Fp}.silu block{i} SiLU", reads="(none)"):
            gate_act = F.silu(gate)
        record(f"{Fp}.silu", LB, i, "SiLU", "gate", gate, "gate_act", gate_act,
               "elementwise SiLU on gate only; up is NOT activated", "(none)",
               "silu(x) = x * sigmoid(x)")

        with Stage(f"{Fp}.up block{i} up projection", reads=f"layer[{i}].mlp.up_proj.weight"):
            up = mlp.up_proj(y_normed)
        record(f"{Fp}.up", LB, i, "up projection", "y_normed", y_normed, "up", up,
               "matmul y_normed @ W_up^T; lifts d -> d_ff (parallel to gate)",
               f"{WEIGHTS_FILE} -> up_proj.weight [{d_ff} x {d}]", "up = y_normed @ W_up")

        with Stage(f"{Fp}.combine block{i} combine", reads="(none)"):
            ffn_hidden = gate_act * up
        record(f"{Fp}.combine", LB, i, "combine (gate_act * up)", "gate_act, up", gate_act,
               "hidden", ffn_hidden, "SwiGLU gating: gate_act modulates up dimension-by-dimension",
               "(none)", "hidden = gate_act (.) up  (Hadamard product)")

        with Stage(f"{Fp}.down block{i} down projection", reads=f"layer[{i}].mlp.down_proj.weight"):
            ffn_delta = mlp.down_proj(ffn_hidden)
        record(f"{Fp}.down", LB, i, "down projection", "hidden", ffn_hidden, "ffn delta", ffn_delta,
               "matmul hidden @ W_down^T; projects d_ff -> d",
               f"{WEIGHTS_FILE} -> down_proj.weight [{d} x {d_ff}]", "ffn_delta = hidden @ W_down")

        with Stage(f"{Fp}.add block{i} add residual (FFN)", reads="(none)"):
            hidden = y_held + ffn_delta
        record(f"{Fp}.add", LB, i, "add residual (FFN)", "y held + ffn delta", ffn_delta,
               "vectors", hidden, "elementwise residual add (pre-norm arch)", "(none)",
               "h = y_held + ffn_delta")

    # ============================ POST-LOOP ============================
    with Stage("post.01 final norm", reads="model.norm.weight"):
        hidden = model.model.norm(hidden)
    record("post.01", OUT, OUT, "final norm", "vectors", hidden, "vectors", hidden,
           "RMSNorm with {final_norm}", f"{WEIGHTS_FILE} -> model.norm.weight [{d}]",
           "y = x / sqrt(mean(x^2)+eps) * weight")

    with Stage("post.02 LM head", reads="lm_head.weight (tied to embed_tokens)"):
        logits_all = model.lm_head(hidden)                 # [1, N_tok, vocab_size]
    record("post.02", OUT, OUT, "LM head", "vectors", hidden, "logits (all pos)", logits_all,
           f"linear projection onto vocab with {{H}} (TIED: {TIED})",
           f"{WEIGHTS_FILE} -> lm_head.weight [{vocab_size} x {d}] (same tensor as embed)",
           "logits = hidden @ H^T ; only step that changes per-token dim")

    with Stage("post.03 take last row", reads="(none)"):
        last_logits = logits_all[0, -1, :]                 # [vocab_size]
    record("post.03", OUT, OUT, "take last row", "logits (all pos)", logits_all,
           "logits (last pos)", last_logits,
           "slice sequence axis at -1; only last position predicts the next token", "(none)",
           "out = logits[0, -1, :]")

    with Stage("post.04 softmax", reads="(none)"):
        probs = torch.softmax(last_logits.float(), dim=-1) # [vocab_size]
    record("post.04", OUT, OUT, "softmax", "logits (last pos)", last_logits, "probabilities", probs,
           "softmax over vocab (fp32); monotonic -> vestigial under greedy argmax", "(none)",
           "p_i = exp(z_i - max z)/sum_j exp(z_j - max z)")

    with Stage("post.05 pick (argmax)", reads="(none)"):
        next_id = torch.argmax(probs, dim=-1)
    record("post.05", OUT, OUT, "pick (argmax)", "probabilities", probs, "token ID", next_id,
           "greedy: index of maximum probability", "(none)", "id = argmax_i p_i")

    with Stage("post.06 decode", reads="tokenizer.json"):
        text_piece = tokenizer.decode(next_id.tolist())
    record("post.06", OUT, OUT, "decode", "token ID", next_id, "text piece", text_piece,
           "inverse vocab lookup + byte-level BPE reassembly (leading space expected)",
           "tokenizer.json (vocab + merges)", "table lookup + byte decode; no FP math")

    return next_id.item(), text_piece, last_logits

# ---------------------------------------------------------------------------
@torch.no_grad()
def verify(prompt: str):
    if TRACE_MODE == "chat":
        messages = [{"role": "user", "content": prompt}]
        enc = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt")
        ids = (enc if torch.is_tensor(enc) else enc["input_ids"]).to(DEVICE)
    else:
        ids = tokenizer(prompt, return_tensors="pt").input_ids.to(DEVICE)
    ref = model(ids).logits[0, -1, :].float()
    my_id, _, my_logits = run_level49(prompt)
    ref_id = torch.argmax(ref).item()
    max_abs = (ref - my_logits.float()).abs().max().item()
    return ref_id, my_id, (ref_id == my_id), max_abs

if __name__ == "__main__":
    tok_id, piece, _ = run_level49("The capital of France is")
    df = pd.DataFrame(STEP_LOG).set_index("id")
    pd.set_option("display.max_colwidth", None); pd.set_option("display.width", None)

    n_pre  = (df["loop"] == "—").sum() - 6          # outer rows minus the 6 post rows
    n_loop = (df["loop"] == "transformer_block").sum()
    print(f"\nsizes: d={d} d_ff={d_ff} n_h={n_h} n_kv={n_kv} d_h={d_h} n_rep={n_rep} "
          f"N={N} vocab={vocab_size} tied={TIED} mode={TRACE_MODE}")
    print(f"rows: total={len(df)}  pre={n_pre}  loop={n_loop} ({n_loop//N} steps x {N} layers)  post=6\n")

    # show layer 0 only, to inspect one iteration of the block
    print("---- one transformer_block iteration (layer 0) ----")
    print(df[df["iter"] == 0][["loop", "iter", "step", "in_shape", "out_shape"]].to_string(), "\n")

    print(f"predicted token id = {tok_id!r}   text = {piece!r}")
    ref_id, my_id, ok, max_abs = verify("The capital of France is")
    print(f"\nverify: reference_argmax={ref_id}  manual_argmax={my_id}  match={ok}  "
          f"max|Δlogit|={max_abs:.3e}")


sizes: d=2048 d_ff=8192 n_h=32 n_kv=8 d_h=64 n_rep=4 N=16 vocab=128256 tied=True mode=completion
rows: total=424  pre=2  loop=416 (26 steps x 16 layers)  post=6

---- one transformer_block iteration (layer 0) ----
                                loop iter                     step           in_shape          out_shape
id                                                                                                      
L00.a.save         transformer_block    0              attn save x     [1 x 6 x 2048]     [1 x 6 x 2048]
L00.a.norm         transformer_block    0            pre-attn norm     [1 x 6 x 2048]     [1 x 6 x 2048]
L00.a.q            transformer_block    0                produce Q     [1 x 6 x 2048]     [1 x 6 x 2048]
L00.a.q_split      transformer_block    0       split Q into heads     [1 x 6 x 2048]  [1 x 32 x 6 x 64]
L00.a.k            transformer_block    0                produce K     [1 x 6 x 2048]      [1 x 6 x 512]
L00.a.k_split      transformer_block    0       sp

In [6]:
df = pd.DataFrame(STEP_LOG)

# one full iteration of the block (all 26 steps for layer 7)
df[df["iter"] == 7]

# the same step-kind across all 16 layers (e.g. every attention-score step)
df[df["step"] == "op: Q.K^T scores"]          # 16 rows, one per layer

# collapse the layer index to see the 26 distinct in-block step-kinds
df_loop = df[df["loop"] == "transformer_block"].copy()
df_loop["kind"] = df_loop["id"].str.replace(r"^L\d+\.", "", regex=True)  # L07.a.q -> a.q
df_loop["kind"].nunique()                      # -> 26

26

In [9]:
# ---------------------------------------------------------------------------
# Total executed steps breakdown + tagged print. Order-based, key-robust:
# does NOT depend on a "#" field. LOOP = steps whose label starts "3.<i>.";
# PRE = non-loop steps before the first loop step; POST = after the last.
# Reuses: run_level49, STEP_LOG, N (=config.num_hidden_layers).
# ---------------------------------------------------------------------------
import re
import pandas as pd

def tag_steps(step_log):
    block_re = re.compile(r"^3\.(\d+)\.")          # "3.0.a.save", "3.15.b.down", ...
    # locate loop region by position
    loop_idx = [j for j, r in enumerate(step_log) if block_re.match(r.get("step", ""))]
    first_loop = loop_idx[0] if loop_idx else len(step_log)
    last_loop  = loop_idx[-1] if loop_idx else -1

    pre, loop, post = [], [], []
    for j, r in enumerate(step_log):
        num = r.get("#", j + 1)                    # fall back to position if "#" absent
        m = block_re.match(r.get("step", ""))
        if m:
            loop.append({**r, "#": num, "region": f"LOOP[{int(m.group(1)):02d}]",
                         "layer": int(m.group(1))})
        elif j < first_loop:
            pre.append({**r, "#": num, "region": "PRE", "layer": None})
        else:  # j > last_loop
            post.append({**r, "#": num, "region": "POST", "layer": None})
    return pre, loop, post

def report_steps(prompt="The capital of France is", show_steps=True):
    run_level49(prompt)                            # repopulate STEP_LOG for this prompt
    pre, loop, post = tag_steps(STEP_LOG)

    n_pre, n_loop, n_post = len(pre), len(loop), len(post)
    total = n_pre + n_loop + n_post
    per_layer = n_loop // N if N else 0

    if show_steps:
        print("=" * 78)
        for r in pre:
            print(f"  {r['#']:>3}  {r['region']:<9}  {r['step']}")
        print("-" * 78)
        for r in loop:
            print(f"  {r['#']:>3}  {r['region']:<9}  {r['step']}")
            if r["step"].endswith("add residual") and ".b." in r["step"]:
                print("-" * 78)                    # separator between layers
        for r in post:
            print(f"  {r['#']:>3}  {r['region']:<9}  {r['step']}")
        print("=" * 78)

    print("\nCOUNT BREAKDOWN")
    print(f"  N (layers / loop iterations)      : {N}")
    print(f"  PRE  (run once)                   : {n_pre}")
    print(f"  LOOP (block body)                 : {n_loop}  = {per_layer} step-types x {N} layers")
    print(f"  POST (run once)                   : {n_post}")
    print(f"  ---------------------------------------------------")
    print(f"  TOTAL executed steps              : {total}")
    print(f"  arithmetic check  {n_pre} + {per_layer}*{N} + {n_post} = {n_pre + per_layer*N + n_post}")
    print(f"  STEP_LOG rows                     : {len(STEP_LOG)}  (match: {total == len(STEP_LOG)})")

    a_types = sum(1 for r in loop if r["layer"] == 0 and ".a." in r["step"])
    b_types = sum(1 for r in loop if r["layer"] == 0 and ".b." in r["step"])
    print(f"\n  block step-types per layer: attention(3.i.a)={a_types}  "
          f"FFN(3.i.b)={b_types}  total={a_types + b_types}")

    return pd.DataFrame(pre + loop + post)

if __name__ == "__main__":
    df_regions = report_steps("The capital of France is", show_steps=True)
    print("\nREGION ROLLUP")
    print(df_regions["region"].str.replace(r"\[\d+\]", "", regex=True)
          .value_counts().reindex(["PRE", "LOOP", "POST"]).to_string())

    1  PRE        tokenize
    2  PRE        embed
    3  PRE        attn save x
    4  PRE        pre-attn norm
    5  PRE        produce Q
    6  PRE        split Q into heads
    7  PRE        produce K
    8  PRE        split K into heads
    9  PRE        produce V
   10  PRE        split V into heads
   11  PRE        rotate Q (RoPE)
   12  PRE        rotate K (RoPE)
   13  PRE        op: Q.K^T scores
   14  PRE        op: scale 1/sqrt(d_h)
   15  PRE        op: causal mask
   16  PRE        op: softmax over keys
   17  PRE        op: weighted sum with V
   18  PRE        concat heads
   19  PRE        output projection
   20  PRE        add residual (attn)
   21  PRE        ffn save y
   22  PRE        pre-FFN norm
   23  PRE        gate projection
   24  PRE        SiLU
   25  PRE        up projection
   26  PRE        combine (gate_act * up)
   27  PRE        down projection
   28  PRE        add residual (FFN)
   29  PRE        attn save x
   30  PRE        pre-attn norm
   3